# SmolVLA Replication + π0.5 Vanilla Instability Backfill

**Prerequisites:** Run [`test_pi05_jennifer.ipynb`](test_pi05_jennifer.ipynb) and [`test_pi05_jennifer_v2.ipynb`](test_pi05_jennifer_v2.ipynb) first so Drive has the package snapshot and π0.5 v2 DB.

**Every session:** Sections 1 → 2 → 3b → 4 → 5 → 6

**Part A (Section 6b):** Re-run π0.5 `vanilla` on the fixed v2 slice and **UPDATE** `action_delta_l2_mean` (and related instability cols) in the existing [`results_v2/rollouts_v2.db`](results_v2/rollouts_v2.db). `SKIP_COMPLETED=True` skips rows that already have instability logged.

**Part B (Section 6c):** Evaluate **SmolVLA** (`HuggingFaceVLA/smolvla_libero`, 450M) on the **same 80 episodes** with `vanilla` + `pnp_uncertainty_only` → [`results_smolvla/rollouts_smolvla.db`](results_smolvla/rollouts_smolvla.db).

**Analysis:** [`pnp_smolvla_jennifer_analysis.ipynb`](pnp_smolvla_jennifer_analysis.ipynb)

> MuJoCo **3.3.2** is recommended for SmolVLA LIBERO eval ([lerobot#1369](https://github.com/huggingface/lerobot/issues/1369)).


---
## Section 1: Drive mount

Run every session.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
SHARED = f'{DRIVE}/cs159-sp26'
CACHE_DIR = f'{DRIVE}/smolvla_colab_cache'
HF_HOME = f'{CACHE_DIR}/hf_models'

PI05_RESULTS_DIR = f'{SHARED}/results_v2'
PI05_V2_DB = f'{PI05_RESULTS_DIR}/rollouts_v2.db'
PI05_VIDEO_DIR = f'{PI05_RESULTS_DIR}/videos_v2'

SMOLVLA_RESULTS_DIR = f'{SHARED}/results_smolvla'
SMOLVLA_DB = f'{SMOLVLA_RESULTS_DIR}/rollouts_smolvla.db'
SMOLVLA_VIDEO_DIR = f'{SMOLVLA_RESULTS_DIR}/videos_smolvla'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(PI05_RESULTS_DIR, exist_ok=True)
os.makedirs(SMOLVLA_RESULTS_DIR, exist_ok=True)
os.makedirs(SMOLVLA_VIDEO_DIR, exist_ok=True)

os.environ['HF_HOME'] = HF_HOME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MUJOCO_GL'] = 'egl'

SNAPSHOT = f'{CACHE_DIR}/site_packages.tar.gz'
print(f'Shared:          {SHARED}')
print(f'π0.5 v2 DB:      {PI05_V2_DB}  (exists={os.path.isfile(PI05_V2_DB)})')
print(f'SmolVLA DB:      {SMOLVLA_DB}')
print(f'Snapshot:        {"FOUND" if os.path.exists(SNAPSHOT) else "NOT FOUND — run test_pi05_jennifer Section 3 first"}')


---
## Section 2: GPU check

In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(out)


---
## Section 3b: Restore packages from Drive snapshot

In [ ]:
import subprocess, sys, os, time, shutil, importlib

if not os.path.exists(SNAPSHOT):
    raise FileNotFoundError('No snapshot — run test_pi05_jennifer Section 3 first.')

LOCAL_SNAPSHOT = '/content/site_packages_restore.tar.gz'
print(f'Copying snapshot ({os.path.getsize(SNAPSHOT)/1e6:.0f} MB)...')
shutil.copy(SNAPSHOT, LOCAL_SNAPSHOT)
subprocess.run(['tar', '-xzf', LOCAL_SNAPSHOT, '-C', '/'], check=True)
os.remove(LOCAL_SNAPSHOT)
importlib.invalidate_caches()

# torch/diffusers compat
try:
    import torch
    from torch.ao.quantization import CUSTOM_KEY  # noqa
    print(f'torch {torch.__version__} OK')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torch', 'torchvision', 'torchaudio'])
    importlib.invalidate_caches()
    import torch
    print(f'torch upgraded to {torch.__version__}')

for pkg in ['mujoco', 'libero', 'lerobot']:
    importlib.import_module(pkg)
    print(f'  {pkg} OK')

try:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    print('SmolVLAPolicy import: OK')
except ImportError:
    print('SmolVLAPolicy missing — installing lerobot[smolvla]...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lerobot[smolvla]'])
    importlib.invalidate_caches()
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    print('SmolVLAPolicy import: OK after install')

from lerobot.policies.pi05.modeling_pi05 import PI05Policy
print('PI05Policy import: OK')


---
## Section 4: LIBERO helpers (policy-agnostic)

In [ ]:
import os, time, json, math
import torch, numpy as np
from huggingface_hub import login
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from lerobot.policies.factory import make_pre_post_processors

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token) if hf_token else login()

MAX_STEPS_MAP = {
    'libero_spatial': 220, 'libero_object': 280, 'libero_goal': 300,
    'libero_10': 520, 'libero_90': 400,
}
CAMERAS = ['agentview', 'robot0_eye_in_hand']
IMG_SIZE = 360
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]
NUM_STEPS_WAIT = 10


def _quat2axisangle(quat):
    if quat[3] > 1.0: quat[3] = 1.0
    elif quat[3] < -1.0: quat[3] = -1.0
    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        return np.zeros(3)
    return (quat[:3] * 2.0 * math.acos(quat[3])) / den


def obs_to_policy(obs_dict, task_desc, device):
    agentview = np.ascontiguousarray(obs_dict['agentview_image'][::-1, ::-1])
    wrist     = np.ascontiguousarray(obs_dict['robot0_eye_in_hand_image'][::-1, ::-1])
    img_agent = torch.from_numpy(agentview / 255.0).permute(2, 0, 1).float()
    img_wrist = torch.from_numpy(wrist / 255.0).permute(2, 0, 1).float()
    state = np.concatenate([
        obs_dict['robot0_eef_pos'],
        _quat2axisangle(obs_dict['robot0_eef_quat']),
        obs_dict['robot0_gripper_qpos'],
    ])
    return {
        'observation.images.image':  img_agent,
        'observation.images.image2': img_wrist,
        'observation.state': torch.from_numpy(state).float(),
        'task': task_desc,
    }

benchmark_dict = benchmark.get_benchmark_dict()
print('Section 4 ready.')


---
## Section 5: P&P sampler, RolloutDB, rollout helpers

In [ ]:
# ── Predict-and-Perturb (P&P) sampler: config, recorder, patched Euler loop ──
import torch, numpy as np
from dataclasses import dataclass
from typing import Optional, Sequence
from lerobot.policies.pi05.modeling_pi05 import make_att_2d_masks


@dataclass
class PnPConfig:
    """Self-refining (Predict-and-Perturb) sampling config for Pi0.5 flow matching.

    LeRobot time runs s=1.0 (noise) -> s=0.0 (clean), so the paper's early/high-noise
    steps are the FIRST Euler steps (large s). Select them with step_indices=(1,)/(1,2)
    or time_min=0.8.

    NOTE on step 0 (s=1.0): the perturb (1-s)*a_hat + s*eps drops a_hat entirely and returns
    fresh noise. So REFINEMENT at s=1.0 is a no-op-like reseed (the map x->eps has no
    contraction). But UNCERTAINTY at s=1.0 is meaningful: each iteration predicts a_hat from
    an independent noise draw, so U measures the spread of the policy's one-shot action
    prediction over the noise prior (overall predictive variance given the observation).
    Default selection starts at step 1 so the *refinement* demo is non-trivial; step 0 is fine
    and informative for mode="uncertainty".
    """
    enabled: bool = False
    step_indices: Optional[Sequence[int]] = (1,)   # which Euler steps run P&P (ignored if time_min set)
    time_min: Optional[float] = None               # alt selector: run P&P when s >= time_min
    num_iterations: int = 3                         # K predict-and-perturb iterations
    mode: str = "both"                              # "uncertainty" | "refine" | "both"
    action_dim: int = 7                             # real (un-padded) action dims used for uncertainty
    record_per_iteration: bool = False              # also store the full (K,B,chunk,adim) a_hat stack per step

    def step_selected(self, step: int, s: float) -> bool:
        if not self.enabled:
            return False
        if self.time_min is not None:
            return s >= self.time_min
        return self.step_indices is not None and step in tuple(self.step_indices)

    @property
    def do_refine(self) -> bool:
        return self.mode in ("refine", "both")


class PnPRecorder:
    """Collects per-episode P&P uncertainty so it can later be correlated with outcomes.

    After a run, `episodes` is a list of dicts:
        {"meta": {...}, "success": bool, "n_steps": int,
         "chunks": [ {"num_steps": int,
                      "steps": [ {"step": i, "s": float,
                                  "u_consecutive": np[B,chunk,adim],  # Eq.10 mean|Δâ|
                                  "a_std": np[B,chunk,adim],          # spread of â over iters
                                  "u_mean", "u_max", "a_std_mean": float,
                                  "u_vec": np[adim],     # per-action-dim mean of u_consecutive
                                  "a_std_vec": np[adim], # per-action-dim mean of a_std
                                  # only if cfg.record_per_iteration:
                                  "a_hats": np[K,B,chunk,adim]}, ... ]}, ... ]}
    One "chunk" == one full action-chunk prediction (one sample_actions call).
    Action dims (LIBERO): 0-2 = xyz pos, 3-5 = axis-angle rot, 6 = gripper.
    """
    def __init__(self):
        self.reset()

    def reset(self):
        self.episodes = []
        self._cur = None

    def new_episode(self, meta=None):
        self._cur = {"meta": dict(meta or {}), "chunks": [], "success": None, "n_steps": None}

    def log_chunk(self, chunk_rec):
        if self._cur is not None:
            self._cur["chunks"].append(chunk_rec)

    def close_episode(self, success, n_steps):
        if self._cur is None:
            return
        self._cur["success"] = bool(success)
        self._cur["n_steps"] = int(n_steps)
        self.episodes.append(self._cur)
        self._cur = None


# Global handles (notebook-style); the patched method reads these each call.
PNP_CONFIG = PnPConfig()
PNP_RECORDER = PnPRecorder()

# v2: optional inference-step override for matched-compute baselines
INFERENCE_NUM_STEPS_OVERRIDE = None


_pnp_disable_compile = getattr(getattr(torch, "compiler", None), "disable", lambda fn: fn)


@_pnp_disable_compile
def _pnp_mark_cuda_graph_step():
    """Tell torch.compile/CUDA graphs that a new policy invocation is starting."""
    mark_step = getattr(getattr(torch, "compiler", None), "cudagraph_mark_step_begin", None)
    if mark_step is not None and torch.cuda.is_available():
        mark_step()


@_pnp_disable_compile
def _pnp_compile_mode(config):
    """Map LeRobot compile modes to CUDA-graph-safe equivalents for P&P sampling."""
    mode = getattr(config, "compile_mode", "default")
    if mode == "max-autotune":
        return "max-autotune-no-cudagraphs"
    return mode


@_pnp_disable_compile
def _pnp_log_chunk(chunk_rec):
    """Side-effect logging must stay outside torch.compile/CUDA graphs."""
    PNP_RECORDER.log_chunk(chunk_rec)


@_pnp_disable_compile
def _pnp_measure_only_actions(model, images, img_masks, tokens, masks, noise, num_steps, kwargs):
    """Run the saved original sampler for uncertainty-only (non-invasive) mode."""
    return model._orig_sample_actions(
        images, img_masks, tokens, masks, noise=noise, num_steps=num_steps, **kwargs
    ).clone()


@_pnp_disable_compile
def _pnp_refine_at_step(x_t, s, vfield, cfg):
    """Run K predict-and-perturb iterations at fixed noise level s.

        predict:  a_hat = x - s * v(x, s)
        perturb:  x'    = (1 - s) * a_hat + s * eps,   eps ~ N(0, I)

    Returns (x_out, rec). x_out is the refined re-noised state if cfg.do_refine, else
    the original x_t unchanged (uncertainty-only is non-invasive). `rec` always holds the
    uncertainty measured across iterations (a free by-product of the predicts).

    This probe does CPU/NumPy logging, so keep it out of torch.compile/CUDA graphs.
    Otherwise graph partitioning and static-buffer reuse can make "uncertainty" mode
    perturb the caller even though it returns the original x_t.
    """
    adim = cfg.action_dim
    x_acc = x_t
    a_hats = []
    for _ in range(cfg.num_iterations):
        v = vfield(x_acc)
        a_hat = x_acc - s * v                       # predicted clean action
        a_hats.append(a_hat[..., :adim])
        eps = torch.randn_like(x_acc)
        x_acc = (1.0 - s) * a_hat + s * eps         # perturb back to level s

    A = torch.stack(a_hats, dim=0)                  # (K, B, chunk, adim)
    if A.shape[0] >= 2:
        u_consecutive = (A[1:] - A[:-1]).abs().mean(dim=0)   # (B, chunk, adim)
        a_std = A.std(dim=0)                                  # (B, chunk, adim)
    else:
        u_consecutive = torch.zeros_like(A[0])
        a_std = torch.zeros_like(A[0])

    # Per-action-dim vectors: mean over batch and chunk → shape (adim,)
    u_vec     = u_consecutive.mean(dim=(0, 1)).detach().float().cpu().numpy()
    a_std_vec = a_std.mean(dim=(0, 1)).detach().float().cpu().numpy()

    rec = {
        "s":             float(s),
        "u_consecutive": u_consecutive.detach().float().cpu().numpy(),
        "a_std":         a_std.detach().float().cpu().numpy(),
        "u_mean":        float(u_consecutive.mean()),
        "u_max":         float(u_consecutive.max()),
        "a_std_mean":    float(a_std.mean()),
        "u_vec":         u_vec,      # np (adim,) — per-dim mean uncertainty
        "a_std_vec":     a_std_vec,  # np (adim,) — per-dim std of predictions
    }
    if cfg.record_per_iteration:
        rec["a_hats"] = A.detach().float().cpu().numpy()
    return (x_acc if cfg.do_refine else x_t), rec


@torch.no_grad()
def _sample_actions_pnp(self, images, img_masks, tokens, masks, noise=None, num_steps=None, **kwargs):
    """Drop-in replacement for PI05Pytorch.sample_actions with optional P&P refinement.

    Delegates to the saved original when P&P is disabled or under RTC; otherwise replicates
    the Euler loop verbatim and injects the P&P inner loop at the selected steps.

    CUDA-graph marks and recorder/logging live in @_pnp_disable_compile helpers *outside*
    this hot path so torch.compile can capture the Euler loop identically to the original.
    Call _pnp_mark_cuda_graph_step() once before each policy invocation (equivalence test,
    rollout, etc.) — not from inside here.
    """
    cfg = PNP_CONFIG
    if (not cfg.enabled) or self._rtc_enabled():
        return self._orig_sample_actions(
            images, img_masks, tokens, masks, noise=noise, num_steps=num_steps, **kwargs)

    if num_steps is None:
        num_steps = INFERENCE_NUM_STEPS_OVERRIDE or self.config.num_inference_steps
    bsize = tokens.shape[0]
    device = tokens.device
    if noise is None:
        actions_shape = (bsize, self.config.chunk_size, self.config.max_action_dim)
        noise = self.sample_noise(actions_shape, device)

    measure_only_output = None
    if cfg.mode == "uncertainty":
        # Guarantee non-invasive behavior: the returned action comes from the saved
        # original sampler, while the custom loop below only populates PNP_RECORDER.
        measure_only_output = _pnp_measure_only_actions(
            self, images, img_masks, tokens, masks, noise.clone(), num_steps, kwargs)

    # ---- prefix / KV cache: replicated verbatim from the original sample_actions ----
    prefix_embs, prefix_pad_masks, prefix_att_masks = self.embed_prefix(images, img_masks, tokens, masks)
    prefix_att_2d_masks = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_position_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1
    prefix_att_2d_masks_4d = self._prepare_attention_masks_4d(prefix_att_2d_masks)
    self.paligemma_with_expert.paligemma.model.language_model.config._attn_implementation = "eager"
    _, past_key_values = self.paligemma_with_expert.forward(
        attention_mask=prefix_att_2d_masks_4d,
        position_ids=prefix_position_ids,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=True,
    )

    dt = -1.0 / num_steps
    x_t = noise
    chunk_rec = {"num_steps": num_steps, "steps": []}

    for step in range(num_steps):
        time = 1.0 + step * dt
        s = time
        time_tensor = torch.tensor(time, dtype=torch.float32, device=device).expand(bsize)

        def denoise_step_partial_call(input_x_t, current_timestep=time_tensor):
            return self.denoise_step(
                prefix_pad_masks=prefix_pad_masks,
                past_key_values=past_key_values,
                x_t=input_x_t,
                timestep=current_timestep,
            )

        if cfg.step_selected(step, s):
            x_t, rec = _pnp_refine_at_step(x_t, s, denoise_step_partial_call, cfg)
            rec["step"] = step
            chunk_rec["steps"].append(rec)

        v_t = denoise_step_partial_call(x_t)
        x_t = x_t + dt * v_t

    _pnp_log_chunk(chunk_rec)
    return measure_only_output if measure_only_output is not None else x_t

_sample_actions_pnp_pi05 = _sample_actions_pnp

from lerobot.policies.smolvla.modeling_smolvla import make_att_2d_masks

@torch.no_grad()
def _sample_actions_pnp_smolvla(self, images, img_masks, lang_tokens, lang_masks, state, noise=None, **kwargs):
    """Drop-in replacement for SmolVLAPytorch.sample_actions with optional P&P."""
    cfg = PNP_CONFIG
    if (not cfg.enabled) or self._rtc_enabled():
        return self._orig_sample_actions(
            images, img_masks, lang_tokens, lang_masks, state, noise=noise, **kwargs)

    num_steps = INFERENCE_NUM_STEPS_OVERRIDE or self.config.num_steps
    bsize = state.shape[0]
    device = state.device
    if noise is None:
        actions_shape = (bsize, self.config.chunk_size, self.config.max_action_dim)
        noise = self.sample_noise(actions_shape, device)

    measure_only_output = None
    if cfg.mode == 'uncertainty':
        measure_only_output = self._orig_sample_actions(
            images, img_masks, lang_tokens, lang_masks, state, noise=noise.clone(), **kwargs
        ).clone()

    prefix_embs, prefix_pad_masks, prefix_att_masks = self.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state)
    prefix_att_2d_masks = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_position_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1
    _, past_key_values = self.vlm_with_expert.forward(
        attention_mask=prefix_att_2d_masks,
        position_ids=prefix_position_ids,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=self.config.use_cache,
        fill_kv_cache=True,
    )

    dt = -1.0 / num_steps
    x_t = noise
    chunk_rec = {'num_steps': num_steps, 'steps': []}

    for step in range(num_steps):
        time = 1.0 + step * dt
        s = time
        time_tensor = torch.tensor(time, dtype=torch.float32, device=device).expand(bsize)

        def denoise_step_partial_call(input_x_t, current_timestep=time_tensor):
            return self.denoise_step(
                prefix_pad_masks=prefix_pad_masks,
                past_key_values=past_key_values,
                x_t=input_x_t,
                timestep=current_timestep,
            )

        if cfg.step_selected(step, s):
            x_t, rec = _pnp_refine_at_step(x_t, s, denoise_step_partial_call, cfg)
            rec['step'] = step
            chunk_rec['steps'].append(rec)

        if self._rtc_enabled():
            v_t = self.rtc_processor.denoise_step(
                x_t=x_t,
                prev_chunk_left_over=kwargs.get('prev_chunk_left_over'),
                inference_delay=kwargs.get('inference_delay'),
                time=time,
                original_denoise_step_partial=denoise_step_partial_call,
                execution_horizon=kwargs.get('execution_horizon'),
            )
        else:
            v_t = denoise_step_partial_call(x_t)
        x_t = x_t + dt * v_t

    _pnp_log_chunk(chunk_rec)
    return measure_only_output if measure_only_output is not None else x_t

print("PnP defined (pi05 + smolvla variants)")


In [ ]:
import types

def _infer_action_dim(policy, default=7):
    feats = getattr(policy.config, 'output_features', {})
    if 'action' in feats:
        return int(feats['action'].shape[0])
    return default


def apply_pnp_patch(policy, flavor='pi05'):
    """Monkey-patch policy.model.sample_actions with P&P wrapper."""
    model = policy.model
    if not hasattr(model, '_orig_sample_actions'):
        model._orig_sample_actions = model.sample_actions
    else:
        model.sample_actions = model._orig_sample_actions

    if flavor == 'pi05':
        fn = _sample_actions_pnp_pi05
        if getattr(model.config, 'compile_model', False):
            compile_mode = _pnp_compile_mode(model.config)
            def _unwrap(fn):
                while hasattr(fn, '_orig_mod'):
                    fn = fn._orig_mod
                return fn
            model._orig_sample_actions = torch.compile(_unwrap(model._orig_sample_actions), mode=compile_mode)
            model.sample_actions = torch.compile(types.MethodType(fn, model), mode=compile_mode)
        else:
            model.sample_actions = types.MethodType(fn, model)
    elif flavor == 'smolvla':
        model.sample_actions = types.MethodType(_sample_actions_pnp_smolvla, model)
    else:
        raise ValueError(flavor)

    PNP_CONFIG.action_dim = _infer_action_dim(policy)
    PNP_CONFIG.enabled = False
    print(f'Patched {flavor} sample_actions (action_dim={PNP_CONFIG.action_dim})')


def load_pi05():
    from lerobot.policies.pi05.modeling_pi05 import PI05Policy
    policy = PI05Policy.from_pretrained('lerobot/pi05_libero_finetuned').to(device).eval()
    preprocess, postprocess = make_pre_post_processors(
        policy.config, 'lerobot/pi05_libero_finetuned',
        preprocessor_overrides={'device_processor': {'device': str(device)}},
    )
    apply_pnp_patch(policy, 'pi05')
    return policy, preprocess, postprocess


def load_smolvla():
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    model_id = 'HuggingFaceVLA/smolvla_libero'
    policy = SmolVLAPolicy.from_pretrained(model_id).to(device).eval()
    preprocess, postprocess = make_pre_post_processors(
        policy.config, model_id,
        preprocessor_overrides={'device_processor': {'device': str(device)}},
    )
    apply_pnp_patch(policy, 'smolvla')
    return policy, preprocess, postprocess


In [ ]:
import os, sqlite3, hashlib, json as _json, time as _time

_ADIM = 7
_U_DIM_COLS    = [f'u_d{i}'     for i in range(_ADIM)]
_ASTD_DIM_COLS = [f'a_std_d{i}' for i in range(_ADIM)]
_DIM_COLS      = _U_DIM_COLS + _ASTD_DIM_COLS
_EXTRA_ROLLOUT_COLS = [
    'method', 'final_eval_slice', 'num_inference_steps', 'num_samples',
    'action_delta_l2_mean', 'action_delta_l2_max', 'action_var_mean',
    'gripper_flip_count', 'gripper_flip_rate', 'chunk_disagreement_mean',
    'policy_model',
]


class RolloutDB:
    """SQLite store for rollout outcomes, P&P uncertainty, and experiment metadata."""

    _DDL = """
    CREATE TABLE IF NOT EXISTS rollouts (
        rollout_id        TEXT PRIMARY KEY,
        suite             TEXT,
        task_idx          INTEGER,
        task_desc         TEXT,
        episode_idx       INTEGER,
        init_state_hash   TEXT,
        success           INTEGER,
        n_steps           INTEGER,
        elapsed_s         REAL,
        pnp_enabled       INTEGER,
        pnp_k             INTEGER,
        pnp_step_indices  TEXT,
        pnp_mode          TEXT,
        u_mean_episode    REAL,
        u_max_episode     REAL,
        n_pnp_activations INTEGER,
        timestamp         TEXT,
        video_path        TEXT,
        method                    TEXT,
        final_eval_slice          INTEGER,
        num_inference_steps       INTEGER,
        num_samples               INTEGER,
        action_delta_l2_mean      REAL,
        action_delta_l2_max       REAL,
        action_var_mean           REAL,
        gripper_flip_count        INTEGER,
        gripper_flip_rate         REAL,
        chunk_disagreement_mean   REAL,
        policy_model              TEXT
    );
    CREATE TABLE IF NOT EXISTS pnp_euler_steps (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        rollout_id   TEXT    NOT NULL REFERENCES rollouts(rollout_id),
        chunk_idx    INTEGER NOT NULL,
        euler_step   INTEGER NOT NULL,
        s            REAL,
        u_mean       REAL,
        u_max        REAL,
        a_std_mean   REAL,
        u_d0 REAL, u_d1 REAL, u_d2 REAL, u_d3 REAL, u_d4 REAL, u_d5 REAL, u_d6 REAL,
        a_std_d0 REAL, a_std_d1 REAL, a_std_d2 REAL, a_std_d3 REAL,
        a_std_d4 REAL, a_std_d5 REAL, a_std_d6 REAL
    );
    CREATE INDEX IF NOT EXISTS idx_pes_rollout ON pnp_euler_steps(rollout_id);
    """

    def __init__(self, db_path):
        self.db_path = str(db_path)
        os.makedirs(os.path.dirname(self.db_path) or '.', exist_ok=True)
        self._con = sqlite3.connect(self.db_path, check_same_thread=False)
        self._con.executescript(self._DDL)
        self._migrate_schema()
        self._con.commit()
        n = self._con.execute('SELECT COUNT(*) FROM rollouts').fetchone()[0]
        print(f"RolloutDB: {self.db_path}  ({n} existing rollouts)")

    def _migrate_schema(self):
        pes_cols = {row[1] for row in self._con.execute("PRAGMA table_info(pnp_euler_steps)")}
        for col in _DIM_COLS:
            if col not in pes_cols:
                self._con.execute(f'ALTER TABLE pnp_euler_steps ADD COLUMN {col} REAL')
        rollout_cols = {row[1] for row in self._con.execute("PRAGMA table_info(rollouts)")}
        type_map = {
            'video_path': 'TEXT', 'method': 'TEXT', 'final_eval_slice': 'INTEGER',
            'num_inference_steps': 'INTEGER', 'num_samples': 'INTEGER',
            'action_delta_l2_mean': 'REAL', 'action_delta_l2_max': 'REAL',
            'action_var_mean': 'REAL', 'gripper_flip_count': 'INTEGER',
            'gripper_flip_rate': 'REAL', 'chunk_disagreement_mean': 'REAL',
            'policy_model': 'TEXT',
        }
        for col, typ in type_map.items():
            if col not in rollout_cols:
                self._con.execute(f'ALTER TABLE rollouts ADD COLUMN {col} {typ}')

    @staticmethod
    def init_state_hash(init_state):
        return hashlib.md5(np.asarray(init_state).tobytes()).hexdigest()[:12]

    @staticmethod
    def make_rollout_id(suite, task_idx, episode_idx, init_state, pnp_cfg,
                        method=None, num_inference_steps=None, num_samples=None,
                        policy_model=None):
        cfg_str = _json.dumps({
            'enabled': pnp_cfg.enabled,
            'k': pnp_cfg.num_iterations,
            'step_indices': list(pnp_cfg.step_indices) if pnp_cfg.step_indices else None,
            'time_min': pnp_cfg.time_min,
            'mode': pnp_cfg.mode,
            'method': method,
            'num_inference_steps': num_inference_steps,
            'num_samples': num_samples,
            'policy_model': policy_model,
        }, sort_keys=True)
        key = f'{suite}:{task_idx}:{episode_idx}:{RolloutDB.init_state_hash(init_state)}:{cfg_str}'
        return hashlib.sha256(key.encode()).hexdigest()[:16]

    def log_episode(self, rollout_id, suite, task_idx, task_desc, episode_idx,
                    init_state, success, n_steps, elapsed_s, pnp_cfg, episode_rec,
                    video_path=None, method=None, final_eval_slice=0,
                    num_inference_steps=None, num_samples=None, instability=None,
                    policy_model=None):
        instability = instability or {}
        all_step_recs = [
            (ci, st)
            for ci, chunk in enumerate(episode_rec.get('chunks', []))
            for st in chunk.get('steps', [])
        ]
        u_vals = [st['u_mean'] for _, st in all_step_recs]
        u_mean_ep = float(np.mean(u_vals)) if u_vals else None
        u_max_ep  = float(np.max(u_vals))  if u_vals else None

        if pnp_cfg.step_indices is not None:
            step_idx_str = _json.dumps(list(pnp_cfg.step_indices))
        else:
            step_idx_str = f'time_min:{pnp_cfg.time_min}' if pnp_cfg.time_min is not None else None

        dim_col_str = ', '.join(_DIM_COLS)
        dim_ph_str  = ', '.join(['?'] * len(_DIM_COLS))

        def _dim_vals(st):
            u_vec     = st.get('u_vec',     [None] * _ADIM)
            a_std_vec = st.get('a_std_vec', [None] * _ADIM)
            return [float(v) if v is not None else None for v in list(u_vec)[:_ADIM]] + \
                   [float(v) if v is not None else None for v in list(a_std_vec)[:_ADIM]]

        with self._con:
            self._con.execute('DELETE FROM pnp_euler_steps WHERE rollout_id = ?', (rollout_id,))
            self._con.execute(
                'INSERT OR REPLACE INTO rollouts VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
                (rollout_id, suite, task_idx, task_desc, episode_idx,
                 self.init_state_hash(init_state),
                 int(success), n_steps, round(elapsed_s, 3),
                 int(pnp_cfg.enabled), pnp_cfg.num_iterations,
                 step_idx_str, pnp_cfg.mode,
                 u_mean_ep, u_max_ep, len(all_step_recs),
                 _time.strftime('%Y-%m-%dT%H:%M:%S'), video_path,
                 method, int(final_eval_slice), num_inference_steps, num_samples,
                 instability.get('action_delta_l2_mean'),
                 instability.get('action_delta_l2_max'),
                 instability.get('action_var_mean'),
                 instability.get('gripper_flip_count'),
                 instability.get('gripper_flip_rate'),
                 instability.get('chunk_disagreement_mean'),
                 policy_model))
            self._con.executemany(
                f'INSERT INTO pnp_euler_steps '
                f'(rollout_id, chunk_idx, euler_step, s, u_mean, u_max, a_std_mean, {dim_col_str}) '
                f'VALUES (?,?,?,?,?,?,?,{dim_ph_str})',
                [(rollout_id, ci, st['step'], st['s'],
                  st['u_mean'], st['u_max'], st['a_std_mean'],
                  *_dim_vals(st))
                 for ci, st in all_step_recs])
        self._con.commit()

    def update_instability(self, rollout_id, instability):
        """Backfill executed-action instability metrics on an existing row."""
        sql = (
            "UPDATE rollouts SET action_delta_l2_mean=?, action_delta_l2_max=?, "
            "action_var_mean=?, gripper_flip_count=?, gripper_flip_rate=?, "
            "chunk_disagreement_mean=? WHERE rollout_id=?"
        )
        with self._con:
            self._con.execute(
                sql,
                (instability.get('action_delta_l2_mean'),
                 instability.get('action_delta_l2_max'),
                 instability.get('action_var_mean'),
                 instability.get('gripper_flip_count'),
                 instability.get('gripper_flip_rate'),
                 instability.get('chunk_disagreement_mean'),
                 rollout_id))
        self._con.commit()

    def vanilla_backfill_targets(self):
        return self.query(
            "SELECT rollout_id, suite, task_idx, episode_idx, init_state_hash "
            "FROM rollouts WHERE method='vanilla' AND final_eval_slice=1 "
            "AND action_delta_l2_mean IS NULL"
        )

    def existing_keys(self, final_eval_slice=1, policy_model=None):
        rows = self.query(
            'SELECT suite, task_idx, episode_idx, init_state_hash, method, pnp_step_indices, policy_model '
            'FROM rollouts WHERE final_eval_slice = ?',
            (int(final_eval_slice),),
        )
        keys = set()
        for r in rows:
            if policy_model is not None and r.get('policy_model') != policy_model:
                continue
            keys.add((r['suite'], r['task_idx'], r['episode_idx'], r['init_state_hash'],
                      r.get('method'), r.get('pnp_step_indices')))
        return keys

    def sync_to_path(self, dst_path):
        import shutil
        tmp = dst_path + '.tmp'
        dst = sqlite3.connect(tmp)
        self._con.backup(dst)
        dst.close()
        shutil.move(tmp, dst_path)
        print(f'Synced DB -> {dst_path}')

    def verify_disk(self, dst_path):
        mem = self._con.execute('SELECT COUNT(*) FROM rollouts').fetchone()[0]
        disk = sqlite3.connect(dst_path).execute('SELECT COUNT(*) FROM rollouts').fetchone()[0]
        print(f'verify: mem={mem} disk={disk}')

    def query(self, sql, params=()):
        cur = self._con.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return [dict(zip(cols, row)) for row in cur.fetchall()]

    def summary(self):
        rows = self.query("""
            SELECT suite, task_idx, method, policy_model,
                   COUNT(*) AS n_ep,
                   ROUND(AVG(success)*100, 1) AS sr_pct,
                   ROUND(AVG(u_mean_episode), 5) AS u_mean_all
            FROM rollouts
            GROUP BY suite, task_idx, method, policy_model
            ORDER BY suite, task_idx, method
        """)
        print(f"{'suite':<18} {'task':>4} {'method':<22} {'model':<8} {'n':>4} {'sr%':>6} {'u_all':>10}")
        print('-' * 82)
        for r in rows:
            print(f"{r['suite']:<18} {r['task_idx']:>4} {str(r.get('method','')):<22} "
                  f"{str(r.get('policy_model') or ''):<8} {r['n_ep']:>4} {r['sr_pct']:>6} {str(r['u_mean_all']):>10}")
        return rows


In [ ]:
# ── Recording-aware rollout + action-instability metrics (v2) ─────────────
import os, imageio

VIDEO_FPS = 10
if 'VIDEO_DIR' not in dir():
    _results = RESULTS_DIR if 'RESULTS_DIR' in dir() else '/content/drive/MyDrive/cs159-sp26/results_v2'
    VIDEO_DIR = os.path.join(_results, 'videos_v2')


def _agentview_frame(obs):
    return np.ascontiguousarray(obs['agentview_image'][::-1, ::-1])


def _compute_chunk_disagreement(chunk_boundary_actions):
    if len(chunk_boundary_actions) < 2:
        return None
    disagreements = [
        float(np.linalg.norm(chunk_boundary_actions[i + 1] - chunk_boundary_actions[i]))
        for i in range(len(chunk_boundary_actions) - 1)
    ]
    return float(np.mean(disagreements))


def _compute_action_instability(executed_actions, chunk_boundary_actions=None, gripper_dim=6, gripper_thresh=0.0):
    if not executed_actions:
        return {
            'action_delta_l2_mean': 0.0,
            'action_delta_l2_max': 0.0,
            'action_var_mean': 0.0,
            'gripper_flip_count': 0,
            'gripper_flip_rate': 0.0,
            'chunk_disagreement_mean': None,
        }
    arr = np.stack([np.asarray(a).flatten()[:getattr(PNP_CONFIG, 'action_dim', 7)] for a in executed_actions])
    if len(arr) >= 2:
        deltas = np.linalg.norm(np.diff(arr, axis=0), axis=1)
        action_delta_l2_mean = float(np.mean(deltas))
        action_delta_l2_max = float(np.max(deltas))
    else:
        action_delta_l2_mean = 0.0
        action_delta_l2_max = 0.0
    action_var_mean = float(np.var(arr, axis=0).mean())
    gripper = arr[:, gripper_dim]
    signs = (gripper > gripper_thresh).astype(int)
    gripper_flip_count = int(np.sum(np.diff(signs) != 0)) if len(signs) > 1 else 0
    gripper_flip_rate = gripper_flip_count / max(len(arr) - 1, 1)
    return {
        'action_delta_l2_mean': action_delta_l2_mean,
        'action_delta_l2_max': action_delta_l2_max,
        'action_var_mean': action_var_mean,
        'gripper_flip_count': gripper_flip_count,
        'gripper_flip_rate': gripper_flip_rate,
        'chunk_disagreement_mean': _compute_chunk_disagreement(chunk_boundary_actions or []),
    }


def _episode_seed(init_state, episode_idx):
    import hashlib as _hs
    _seed_bytes = _hs.md5(
        np.asarray(init_state).tobytes() + str(episode_idx or 0).encode()
    ).digest()
    return int.from_bytes(_seed_bytes[:4], 'big')


def _to_numpy_action(action):
    action = postprocess(action)
    if isinstance(action, torch.Tensor):
        action = action.squeeze(0).cpu().numpy()
    return np.asarray(action).flatten()


def run_episode_pnp(env, init_state, policy, task_desc, max_steps, device,
                    ep_meta=None, db=None,
                    suite=None, task_idx=None, episode_idx=None,
                    save_video=False,
                    method=None, final_eval_slice=0,
                    num_inference_steps=None, num_samples=None):
    env.reset(); policy.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)

    _seed = _episode_seed(init_state, episode_idx)
    torch.manual_seed(_seed)
    torch.cuda.manual_seed(_seed)

    global INFERENCE_NUM_STEPS_OVERRIDE
    _prev_override = INFERENCE_NUM_STEPS_OVERRIDE
    if num_inference_steps is not None:
        INFERENCE_NUM_STEPS_OVERRIDE = num_inference_steps

    PNP_RECORDER.new_episode(ep_meta)
    record_video = save_video in (True, 'failures_only')
    frames = [] if record_video else None
    executed_actions = []
    chunk_boundary_actions = []
    last_n_chunks = 0

    rollout_id = None
    if suite is not None:
        rollout_id = RolloutDB.make_rollout_id(
            suite, task_idx or 0, episode_idx or 0, init_state, PNP_CONFIG,
            method=method, num_inference_steps=num_inference_steps, num_samples=num_samples,
            policy_model=globals().get('CURRENT_POLICY_MODEL'))

    t0 = time.time(); success = False; step = 0
    try:
        for step in range(max_steps):
            if record_video:
                frames.append(_agentview_frame(obs))
            raw_obs = obs_to_policy(obs, task_desc, device)
            batch = preprocess(raw_obs)
            _pnp_mark_cuda_graph_step()
            with torch.no_grad():
                action = policy.select_action(batch)
            action_np = _to_numpy_action(action)
            executed_actions.append(action_np.copy())
            n_chunks = len(PNP_RECORDER._cur['chunks']) if PNP_RECORDER._cur else 0
            if n_chunks > last_n_chunks:
                chunk_boundary_actions.append(action_np.copy())
                last_n_chunks = n_chunks
            obs, _, done, _ = env.step(action_np)
            if env.check_success():
                success = True
                break
            if done:
                break
    finally:
        INFERENCE_NUM_STEPS_OVERRIDE = _prev_override

    elapsed = time.time() - t0
    PNP_RECORDER.close_episode(success, step + 1)
    instability = _compute_action_instability(executed_actions, chunk_boundary_actions)

    video_path = None
    should_save = save_video is True or (save_video == 'failures_only' and not success)
    if record_video and should_save and frames and rollout_id is not None:
        os.makedirs(VIDEO_DIR, exist_ok=True)
        video_path = os.path.join(VIDEO_DIR, f'{rollout_id}.mp4')
        imageio.mimsave(video_path, frames, fps=VIDEO_FPS)

    if db is not None:
        db.log_episode(
            rollout_id=rollout_id,
            suite=suite or '',
            task_idx=task_idx or 0,
            task_desc=task_desc,
            episode_idx=episode_idx or 0,
            init_state=init_state,
            success=success,
            n_steps=step + 1,
            elapsed_s=elapsed,
            pnp_cfg=PNP_CONFIG,
            episode_rec=PNP_RECORDER.episodes[-1],
            video_path=video_path,
            method=method,
            final_eval_slice=final_eval_slice,
            num_inference_steps=num_inference_steps,
            num_samples=num_samples,
            instability=instability,
            policy_model=globals().get('CURRENT_POLICY_MODEL'),
        )

    return success, step + 1, elapsed


def _multi_sample_chunk(policy, batch, base_seed, chunk_idx, num_samples, probe_steps):
    """Sample num_samples chunks; probe U at probe_steps; return lowest-U chunk."""
    saved = (PNP_CONFIG.enabled, PNP_CONFIG.mode, PNP_CONFIG.step_indices, PNP_CONFIG.num_iterations)
    PNP_CONFIG.enabled = True
    PNP_CONFIG.mode = 'uncertainty'
    PNP_CONFIG.step_indices = probe_steps
    PNP_CONFIG.num_iterations = globals().get('PNP_K', 3)
    best_chunk = None
    best_u = float('inf')
    best_chunks = None
    chunk_start = len(PNP_RECORDER._cur['chunks']) if PNP_RECORDER._cur else 0
    for si in range(num_samples):
        policy.reset()  # fresh KV/cache per candidate
        torch.manual_seed(base_seed + chunk_idx * 1000 + si)
        torch.cuda.manual_seed(base_seed + chunk_idx * 1000 + si)
        _pnp_mark_cuda_graph_step()
        with torch.no_grad():
            chunk = policy.predict_action_chunk(batch, noise=None).clone()
        new_chunks = PNP_RECORDER._cur['chunks'][chunk_start:] if PNP_RECORDER._cur else []
        u_vals = [st['u_mean'] for c in new_chunks for st in c.get('steps', [])]
        u_score = float(np.mean(u_vals)) if u_vals else float('inf')
        if u_score < best_u:
            best_u = u_score
            best_chunk = chunk
            best_chunks = list(new_chunks)
        if PNP_RECORDER._cur is not None:
            PNP_RECORDER._cur['chunks'] = PNP_RECORDER._cur['chunks'][:chunk_start]
    if PNP_RECORDER._cur is not None and best_chunks is not None:
        PNP_RECORDER._cur['chunks'].extend(best_chunks)
    PNP_CONFIG.enabled, PNP_CONFIG.mode, PNP_CONFIG.step_indices, PNP_CONFIG.num_iterations = saved
    if best_chunk is None:
        policy.reset()
        _pnp_mark_cuda_graph_step()
        with torch.no_grad():
            return policy.predict_action_chunk(batch, noise=None)
    return best_chunk


def run_episode_multi_sample(env, init_state, policy, task_desc, max_steps, device,
                             ep_meta=None, db=None,
                             suite=None, task_idx=None, episode_idx=None,
                             save_video=False, num_samples=3, probe_steps=(2, 3),
                             method='multi_sample_select', final_eval_slice=0):
    env.reset(); policy.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)

    _seed = _episode_seed(init_state, episode_idx)
    torch.manual_seed(_seed)
    torch.cuda.manual_seed(_seed)
    PNP_RECORDER.new_episode(ep_meta)
    record_video = save_video in (True, 'failures_only')
    frames = [] if record_video else None
    executed_actions = []
    chunk_boundary_actions = []
    action_queue = []
    chunk_idx = 0

    PNP_CONFIG.enabled = False
    _saved_probe = PNP_CONFIG.step_indices
    PNP_CONFIG.step_indices = probe_steps
    rollout_id = RolloutDB.make_rollout_id(
        suite or '', task_idx or 0, episode_idx or 0, init_state, PNP_CONFIG,
        method=method, num_samples=num_samples) if suite is not None else None
    PNP_CONFIG.step_indices = _saved_probe

    t0 = time.time(); success = False; step = 0
    for step in range(max_steps):
        if record_video:
            frames.append(_agentview_frame(obs))
        if not action_queue:
            raw_obs = obs_to_policy(obs, task_desc, device)
            batch = preprocess(raw_obs)
            chunk = _multi_sample_chunk(policy, batch, _seed, chunk_idx, num_samples, probe_steps)
            chunk_idx += 1
            chunk_np = chunk.squeeze(0).cpu().numpy()
            for i in range(chunk_np.shape[0]):
                action_queue.append(chunk_np[i].copy())
            chunk_boundary_actions.append(action_queue[0].copy())
        action_np = action_queue.pop(0)
        executed_actions.append(action_np.copy())
        obs, _, done, _ = env.step(action_np)
        if env.check_success():
            success = True
            break
        if done:
            break

    elapsed = time.time() - t0
    PNP_RECORDER.close_episode(success, step + 1)
    instability = _compute_action_instability(executed_actions, chunk_boundary_actions)

    video_path = None
    should_save = save_video is True or (save_video == 'failures_only' and not success)
    if record_video and should_save and frames and rollout_id is not None:
        os.makedirs(VIDEO_DIR, exist_ok=True)
        video_path = os.path.join(VIDEO_DIR, f'{rollout_id}.mp4')
        imageio.mimsave(video_path, frames, fps=VIDEO_FPS)

    if db is not None:
        db.log_episode(
            rollout_id=rollout_id,
            suite=suite or '',
            task_idx=task_idx or 0,
            task_desc=task_desc,
            episode_idx=episode_idx or 0,
            init_state=init_state,
            success=success,
            n_steps=step + 1,
            elapsed_s=elapsed,
            pnp_cfg=PNP_CONFIG,
            episode_rec=PNP_RECORDER.episodes[-1],
            video_path=video_path,
            method=method,
            final_eval_slice=final_eval_slice,
            num_inference_steps=None,
            num_samples=num_samples,
            instability=instability,
            policy_model=globals().get('CURRENT_POLICY_MODEL'),
        )

    return success, step + 1, elapsed


CURRENT_POLICY_MODEL = None

def run_episode_backfill(env, init_state, policy, task_desc, max_steps, device,
                         episode_idx=None):
    # Run vanilla episode without DB write; return instability dict only.
    env.reset(); policy.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)
    _seed = _episode_seed(init_state, episode_idx)
    torch.manual_seed(_seed); torch.cuda.manual_seed(_seed)
    PNP_CONFIG.enabled = False
    PNP_RECORDER.new_episode({})
    executed_actions, chunk_boundary_actions = [], []
    last_n_chunks = 0
    for step in range(max_steps):
        raw_obs = obs_to_policy(obs, task_desc, device)
        batch = preprocess(raw_obs)
        _pnp_mark_cuda_graph_step()
        with torch.no_grad():
            action = policy.select_action(batch)
        action_np = _to_numpy_action(action)
        executed_actions.append(action_np.copy())
        n_chunks = len(PNP_RECORDER._cur['chunks']) if PNP_RECORDER._cur else 0
        if n_chunks > last_n_chunks:
            chunk_boundary_actions.append(action_np.copy())
            last_n_chunks = n_chunks
        obs, _, done, _ = env.step(action_np)
        if env.check_success() or done:
            break
    return _compute_action_instability(executed_actions, chunk_boundary_actions)


---
## Section 6: Controlled experiment

**Flags:**
- `RUN_PI05_BACKFILL` — Part A: fill π0.5 vanilla instability in existing v2 DB
- `RUN_SMOLVLA_EVAL` — Part B: SmolVLA vanilla + detector on same slice
- `SKIP_COMPLETED` — skip episode×method combos already present (SmolVLA DB) or rows with non-null instability (backfill)


In [ ]:
import sqlite3

FINAL_STEP_CONFIGS = [(2, 3), (3, 4), (4, 5)]
FINAL_EPISODE_IDXS = list(range(10))
PNP_K = 3
BASELINE_STEPS = 10

RUN_PI05_BACKFILL = True
RUN_SMOLVLA_EVAL = True
SKIP_COMPLETED = True

SMOLVLA_METHODS = ['vanilla', 'pnp_uncertainty_only']
RUN_SMOLVLA_METHODS = ['vanilla', 'pnp_uncertainty_only']

# Hardcoded fallback (neurips appendix v2 slice)
FALLBACK_TASKS = [
    ('libero_spatial', 5), ('libero_spatial', 8),
    ('libero_goal', 0), ('libero_goal', 1), ('libero_goal', 2),
    ('libero_goal', 3), ('libero_goal', 5), ('libero_goal', 6),
]

FINAL_EPISODES = []
EPISODE_KEYS = set()
_db_keys = set()

if os.path.isfile(PI05_V2_DB):
    _con = sqlite3.connect(PI05_V2_DB)
    rows = _con.execute(
        "SELECT DISTINCT suite, task_idx, episode_idx, init_state_hash "
        "FROM rollouts WHERE final_eval_slice=1 AND method='vanilla'"
    ).fetchall()
    _con.close()
    _db_keys = {(r[0], r[1], r[2], r[3]) for r in rows}
    task_set = sorted({(r[0], r[1]) for r in rows}) if rows else FALLBACK_TASKS
    print(f'Loaded {len(rows)} vanilla keys from π0.5 v2 DB -> {len(task_set)} tasks')
else:
    task_set = FALLBACK_TASKS
    print('π0.5 v2 DB not found — using fallback task list')

for suite, task_idx in task_set:
    task_suite = benchmark_dict[suite]()
    task = task_suite.get_task(task_idx)
    init_states = task_suite.get_task_init_states(task_idx)
    bddl_path = os.path.join(get_libero_path('bddl_files'), task.problem_folder, task.bddl_file)
    max_steps = MAX_STEPS_MAP.get(suite, 300)
    for ep_idx in FINAL_EPISODE_IDXS:
        if ep_idx >= len(init_states):
            continue
        init_state = init_states[ep_idx]
        ish = RolloutDB.init_state_hash(init_state)
        key = (suite, task_idx, ep_idx, ish)
        if _db_keys and key not in _db_keys:
            continue
        EPISODE_KEYS.add(key)
        FINAL_EPISODES.append(dict(
            suite=suite, task_idx=task_idx, task_desc=task.language,
            ep_idx=ep_idx, init_state=init_state, bddl_path=bddl_path,
            max_steps=max_steps, init_state_hash=ish,
        ))

print(f'FINAL_EPISODES: {len(FINAL_EPISODES)}')


In [ ]:
# === Part A: π0.5 vanilla instability backfill ===
from itertools import groupby
from tqdm.notebook import tqdm

if RUN_PI05_BACKFILL:
    if not os.path.isfile(PI05_V2_DB):
        print('Skip backfill — π0.5 v2 DB not found')
    else:
        pi05_db = RolloutDB(PI05_V2_DB)
        targets = pi05_db.vanilla_backfill_targets()
        print(f'Backfill targets (NULL action_delta_l2_mean): {len(targets)}')
        if not targets:
            print('Nothing to backfill — all vanilla rows have instability metrics.')
        else:
            target_map = {(r['suite'], r['task_idx'], r['episode_idx'], r['init_state_hash']): r['rollout_id']
                          for r in targets}
            policy, preprocess, postprocess = load_pi05()
            CURRENT_POLICY_MODEL = 'pi05'
            PNP_CONFIG.enabled = False
            sorted_eps = sorted(FINAL_EPISODES, key=lambda x: (x['suite'], x['task_idx']))
            n_updated = 0
            for (suite, task_idx), group_iter in groupby(sorted_eps, key=lambda x: (x['suite'], x['task_idx'])):
                episodes = list(group_iter)
                env = OffScreenRenderEnv(
                    bddl_file_name=episodes[0]['bddl_path'], camera_names=CAMERAS,
                    camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
                    has_offscreen_renderer=True, use_camera_obs=True,
                    has_renderer=False, reward_shaping=False,
                )
                try:
                    for ep in tqdm(episodes, desc=f'backfill {suite} T{task_idx}', leave=False):
                        key = (ep['suite'], ep['task_idx'], ep['ep_idx'], ep['init_state_hash'])
                        if key not in target_map:
                            continue
                        rid = target_map[key]
                        inst = run_episode_backfill(
                            env, ep['init_state'], policy, ep['task_desc'], ep['max_steps'],
                            device, episode_idx=ep['ep_idx'])
                        pi05_db.update_instability(rid, inst)
                        n_updated += 1
                finally:
                    env.close()
            pi05_db.sync_to_path(PI05_V2_DB)
            print(f'Updated instability on {n_updated} vanilla rows')
            pi05_db.summary()
else:
    print('RUN_PI05_BACKFILL=False — skipped')


In [ ]:
# === Part B: SmolVLA eval (vanilla + pnp_uncertainty_only) ===
import json as _json
from itertools import groupby
from tqdm.notebook import tqdm

if RUN_SMOLVLA_EVAL:
    policy, preprocess, postprocess = load_smolvla()
    CURRENT_POLICY_MODEL = 'smolvla'
    VIDEO_DIR = SMOLVLA_VIDEO_DIR
    DB = RolloutDB(SMOLVLA_DB)

    if not FINAL_EPISODES:
        print('No FINAL_EPISODES — run slice cell first')
    else:
        sorted_eps = sorted(FINAL_EPISODES, key=lambda x: (x['suite'], x['task_idx']))
        for method in RUN_SMOLVLA_METHODS:
            print(f'\n{"#"*60}\nSmolVLA method: {method}\n{"#"*60}')
            configs = list(FINAL_STEP_CONFIGS) if method == 'pnp_uncertainty_only' else [None]
            for step_indices in configs:
                PNP_CONFIG.enabled = (method == 'pnp_uncertainty_only')
                PNP_CONFIG.mode = 'uncertainty'
                PNP_CONFIG.step_indices = step_indices
                PNP_CONFIG.num_iterations = PNP_K
                PNP_CONFIG.time_min = None
                PNP_RECORDER.reset()
                step_key = _json.dumps(list(step_indices)) if step_indices else None
                completed = DB.existing_keys(policy_model='smolvla') if SKIP_COMPLETED else set()

                for (suite, task_idx), group_iter in groupby(sorted_eps, key=lambda x: (x['suite'], x['task_idx'])):
                    episodes = list(group_iter)
                    env = OffScreenRenderEnv(
                        bddl_file_name=episodes[0]['bddl_path'], camera_names=CAMERAS,
                        camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
                        has_offscreen_renderer=True, use_camera_obs=True,
                        has_renderer=False, reward_shaping=False,
                    )
                    try:
                        for ep in tqdm(episodes, desc=f'  {method} {suite} T{task_idx}', leave=False):
                            ep_key = (ep['suite'], ep['task_idx'], ep['ep_idx'], ep['init_state_hash'],
                                      method, step_key)
                            if ep_key in completed:
                                continue
                            run_episode_pnp(
                                env, ep['init_state'], policy, ep['task_desc'], ep['max_steps'], device,
                                suite=ep['suite'], task_idx=ep['task_idx'], episode_idx=ep['ep_idx'],
                                db=DB, save_video='failures_only', method=method, final_eval_slice=1,
                                num_inference_steps=BASELINE_STEPS,
                            )
                    finally:
                        env.close()
        DB.sync_to_path(SMOLVLA_DB)
        DB.summary()
else:
    print('RUN_SMOLVLA_EVAL=False — skipped')


In [ ]:
# Optional: SmolVLA equivalence smoke test (P&P off)
if RUN_SMOLVLA_EVAL and FINAL_EPISODES:
    policy, preprocess, postprocess = load_smolvla()
    ep = FINAL_EPISODES[0]
    env = OffScreenRenderEnv(
        bddl_file_name=ep['bddl_path'], camera_names=CAMERAS,
        camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
        has_offscreen_renderer=True, use_camera_obs=True, has_renderer=False, reward_shaping=False,
    )
    try:
        env.reset(); policy.reset()
        obs = env.set_init_state(ep['init_state'])
        for _ in range(NUM_STEPS_WAIT):
            obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)
        batch = preprocess(obs_to_policy(obs, ep['task_desc'], device))
        PNP_CONFIG.enabled = False
        torch.manual_seed(0); torch.cuda.manual_seed(0)
        with torch.no_grad():
            a1 = policy.predict_action_chunk(batch, noise=None).clone()
        PNP_CONFIG.enabled = True
        PNP_CONFIG.step_indices = ()
        PNP_CONFIG.mode = 'uncertainty'
        torch.manual_seed(0); torch.cuda.manual_seed(0)
        with torch.no_grad():
            a2 = policy.predict_action_chunk(batch, noise=None).clone()
        print(f'SmolVLA loop equiv max|Δ|={ (a1-a2).abs().max().item():.3e}')
    finally:
        env.close()
